In [1]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, Set, List
from collections import defaultdict


## Drugs @ FDA

In [3]:
fda_data_dir = "./data"
out_dir = "./out"

In [2]:
fda_drugs_file = f"{fda_data_dir}/drug-drugsfda-0001-of-0001.json"

In [3]:

# Load the FDA JSON file
with open(fda_drugs_file, "r") as f:
    data = json.load(f)

# Initialize list for filtered results
orig_records = []

# Loop over results
for result in data.get("results", []):
    submissions = result.get("submissions", [])
    
    # Find the ORIG submission(s)
    orig_subs = [s for s in submissions if s.get("submission_type") == "ORIG"]
    if not orig_subs:
        continue

    orig = orig_subs[0]
    submission_date = orig.get("submission_status_date")
    year = submission_date[:4] if submission_date else None
    # Extract base info
    record = {
        "application_number": result.get("application_number"),
        "sponsor_name": result.get("sponsor_name"),
        "submission_type": orig.get("submission_type"),
        "submission_status": orig.get("submission_status"),
        "submission_status_date": submission_date,
        "submission_class_code_description": orig.get("submission_class_code_description"),
        "year": year
    }

    # Extract first product if exists
    products = result.get("products", [])
    if products:
        p = products[0]
        record.update({
            "brand_name": p.get("brand_name"),
            "dosage_form": p.get("dosage_form"),
            "route": p.get("route"),
            "marketing_status": p.get("marketing_status"),
            "active_ingredients": ", ".join(
                f"{a.get('name')}" for a in p.get("active_ingredients", [])
            )
        })

    # If openfda section exists, flatten a few key fields
    openfda = result.get("openfda", {})
    if openfda:
        record.update({
            "openfda_brand_name": ", ".join(openfda.get("brand_name", [])),
            "openfda_generic_name": ", ".join(openfda.get("generic_name", [])),
            "openfda_route": ", ".join(openfda.get("route", [])),
            "openfda_substance_name": ", ".join(openfda.get("substance_name", []))
        })

    orig_records.append(record)

# Convert to DataFrame for easier viewing/analysis
df_orig = pd.DataFrame(orig_records)

In [4]:
df_orig.shape

(25712, 16)

In [5]:
df_orig.submission_status.value_counts()

submission_status
AP    24636
TA     1075
Name: count, dtype: int64

## Label

In [21]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# --- only these columns will be written into df_orig
label_cols = [
    "label_brand_name",
    "label_generic_name",
    "label_manufacturer_name",
    "label_substance_name",
    "indications_first_sent",
    "indications_and_usage",
    "clinical_studies",
]

# ensure df_orig has target columns (so .update works cleanly)
for col in label_cols:
    if col not in df_orig.columns:
        df_orig[col] = pd.NA

# regex: first sentence without splitting on numbered list markers like "1."
FIRST_SENTENCE_RE = re.compile(r"^(.*?)(?<!\d)\.\s")

def first_sentence(text: str) -> str:
    if not text:
        return ""
    m = FIRST_SENTENCE_RE.match(text)
    return (m.group(1) if m else text).strip()

# --- iterate files 01..13, process and merge immediately
base_dir = Path(fda_data_dir)

for i in tqdm(range(1, 14), desc="Processing label files", unit="file"):
    file_nr = f"{i:02d}"
    drug_label_file = base_dir / f"drug-label-00{file_nr}-of-0013.json"

    with open(drug_label_file, "r") as f:
        data = json.load(f)

    results = data.get("results", [])

    # build minimal records for THIS file only
    records = []
    for result in tqdm(results, desc=f"Parsing {drug_label_file.name}", unit="label", leave=False):
        openfda = result.get("openfda", {}) or {}
        application_numbers = openfda.get("application_number", []) or []

        # Skip if no application_number
        if not application_numbers:
            continue

        # clinical trials: tolerant NCT regex (as you wrote)
        clinical_text = " ".join(result.get("clinical_studies", []) or [])
        nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

        # indications: first sentence only (don’t split on "1.")
        indications_text = " ".join(result.get("indications_and_usage", []) or [])
        match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
        first_sent = match.group(1).strip() if match else indications_text.strip()

        # one row per application_number (clean merge key)
        for app_no in application_numbers:
            records.append({
                "application_number": app_no,
                "label_brand_name": ", ".join(openfda.get("brand_name", []) or []),
                "label_generic_name": ", ".join(openfda.get("generic_name", []) or []),
                "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", []) or []),
                "label_substance_name": ", ".join(openfda.get("substance_name", []) or []),
                "indications_first_sent": first_sent,
                "indications_and_usage": indications_text,
                "clinical_studies": nct_ids,  # keep as list
            })

    # turn this file's records into a df, dedupe by application_number to avoid row explosion
    df_labels_file = pd.DataFrame(records)
    if not df_labels_file.empty:
        df_labels_file = df_labels_file.drop_duplicates(subset=["application_number"], keep="first")
        # in-place update (no _x/_y columns, no row duplication)
        df_orig = df_orig.set_index("application_number")
        df_orig.update(df_labels_file.set_index("application_number")[label_cols])
        df_orig = df_orig.reset_index()

print("✅ Finished merging each label file into df_orig (on the fly).")


Processing label files: 100%|██████████| 13/13 [00:35<00:00,  2.76s/file]                       

✅ Finished merging each label file into df_orig (on the fly).


In [23]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"].to_csv("out/CLADRIBINE_drugs_labels_full.csv",index=False)

In [14]:
df_orig.to_csv(f"{out_dir}/FDA_drugs_labels_full.csv",index=False)


### embed drug terms (for merge with target DS and parsing of indications)

In [15]:
df_orig = pd.read_csv(f"{out_dir}/FDA_drugs_labels_full.csv")

In [16]:
melatonin_rows = df_orig[
    df_orig
    .astype(str)
    .apply(lambda col: col.str.contains("amisulpride", case=False, na=False))
    .any(axis=1)
]
melatonin_rows

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
9302,NDA209510,ACACIA,ORIG,AP,20200226,Type 1 - New Molecular Entity,2020,BARHEMSYS,SOLUTION,INTRAVENOUS,...,AMISULPRIDE,INTRAVENOUS,AMISULPRIDE,Barhemsys,AMISULPRIDE,Acacia Pharma Ltd,AMISULPRIDE,1 INDICATIONS AND USAGE BARHEMSYS ® is indicat...,1 INDICATIONS AND USAGE BARHEMSYS ® is indicat...,"['NCT01991860', 'NCT02337062', 'NCT02449291', ..."


### select approvals

In [17]:
df_orig = df_orig[df_orig['submission_status']=="AP"]
df_orig.shape

(25230, 23)

In [18]:
cols = ["active_ingredients", "label_substance_name", "brand_name"]

df_orig["active_ingredients_substance_brand"] = (
    df_orig[cols]
    .fillna("")
    .apply(
        lambda row: ", ".join(
            dict.fromkeys(  # preserves order while removing duplicates
                v.strip() for v in row if v.strip()
            )
        ),
        axis=1
    )
)


In [19]:
terms = df_orig[['active_ingredients','label_substance_name','active_ingredients_substance_brand']]
terms[terms['label_substance_name']=="CARIPRAZINE"]

,active_ingredients,label_substance_name,active_ingredients_substance_brand
21966,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE,"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR"


In [20]:
df_orig = df_orig[
    df_orig["active_ingredients_substance_brand"].notna() &
    (df_orig["active_ingredients_substance_brand"].astype(str).str.strip() != "")
]

df_orig.shape

(25216, 24)

In [21]:
df_orig["active_ingredients_split"] = (
    df_orig["active_ingredients_substance_brand"]
    .astype(str)
    .str.split(r"(?i),\s*|\|\||\s*\band\b\s*", regex=True)
    .apply(
        lambda lst: list(
            dict.fromkeys(  # de-dupe, keep order
                s.strip() for s in lst if s and s.strip()
            )
        )
    )
)

# Explode into separate rows
df_orig = df_orig.explode("active_ingredients_split", ignore_index=True)

df_orig.shape

(38900, 25)

In [24]:
df_orig.to_csv(f"{out_dir}/FDA_drugs_labels_full_AP.csv",index=False)

### map flat ingredients to ontology

In [25]:
import sys
sys.path.append("./entity_linking")   # adjust path to your real folder
from neural_based_nen import main

In [26]:
data_dir =  "./entity_linking/data/"
mapping_type = "drug"
col_to_map = "active_ingredients_split"
input_file = f"{out_dir}/FDA_drugs_labels_full_AP.csv"
output_file = f"{out_dir}/FDA_drugs_labels_full_AP_mapped_active_ingredients.csv"

main(mapping_type, col_to_map, data_dir, input_file, output_file, stats_dir=None)

Input file: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP.csv


/home/sdonev/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using terminology: umls
Using distance threshold: 8.2
Starting normalization for: DRUG with cdist 8.2
Loading embeddings from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/embeddings with prefix UMLS_emb...
Loading term-id pairs from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/umls_term_id_pairs_combined.json...
Found COMBINED embeddings file, loading that one...
Loaded embeddings: torch.Size([1315534, 768]), term_id_pairs: 1315534, canonical mappings: 474316
Found 8401 unique terms in 'active_ingredients_split'


Mapping active_ingredients_split NER to umls: 100%|██████████| 38900/38900 [00:04<00:00, 9470.99it/s] 


Normalization time for 'active_ingredients_split': 0:26:57
Output saved to: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP_mapped_active_ingredients.csv


In [34]:
fda_embedded_full = pd.read_csv(output_file)
fda_embedded_full.head()

,active_ingredients_split,linkbert_umls_split,drug_umls_termid,drug_umls_term_norm,drug_umls_closest_3,drug_umls_cdist
0,HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261,Hydrochlorothiazide,"[['hydrochlorothiazide', 'C0020261'], ['Hydroc...",0.0055
1,LISINOPRIL,Lisinopril,C0065374,Lisinopril,"[['lisinopril', 'C0065374'], ['Lisinopril', 'C...",0.0000
2,CALCITRIOL,Calcitriol,C0006674,Calcitriol,"[['calcitriol', 'C0006674'], ['CALCITRIOL', 'C...",0.0000
3,CARBIDOPA,Carbidopa,C0006982,Carbidopa,"[['Carbidopa', 'C0006982'], ['carbidopa', 'C00...",0.0078
4,LEVODOPA,Levodopa,C0023570,Levodopa,"[['Levodopa', 'C0023570'], ['levodopa', 'C0023...",0.0110


In [35]:
fda_embedded_full.shape

(38900, 6)

In [36]:
fda_embedded_with_details = pd.concat([
    df_orig.reset_index(drop=True), 
    fda_embedded_full[['drug_umls_term_norm', 'drug_umls_termid']].reset_index(drop=True)
], axis=1)
fda_embedded_with_details.head()

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261
1,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",LISINOPRIL,Lisinopril,C0065374
2,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,CALCITRIOL,CALCITRIOL,Calcitriol,C0006674
3,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",CARBIDOPA,Carbidopa,C0006982
4,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",LEVODOPA,Levodopa,C0023570


In [37]:
fda_embedded_with_details.shape

(38900, 27)

In [38]:
fda_embedded_with_details.to_csv(f"{out_dir}/FDA_full_AP_mapped_active_ingredients.csv",index=False)

### add parents terminology
from ../04/UMLS_Drug_Parent

In [42]:
fda_embedded_with_details = pd.read_csv(f"{out_dir}/FDA_full_AP_mapped_active_ingredients.csv")

In [43]:
import json

with open("./entity_linking/parent_mappings/umls_cui_set_full_dataset.json", "r") as f:
    all_umls_ids = set(json.load(f))

with open("./entity_linking/parent_mappings/umls_cui2_to_cui1_mapping.json", "r") as f:
    cui2_to_cui1 = json.load(f)

with open("./entity_linking/parent_mappings/umls_cui_to_str.json", "r") as f:
    cui_to_str = json.load(f)

with open("./entity_linking/parent_mappings/parent_counts.json", "r") as f:
    parent_counts = json.load(f)

In [44]:
def get_all_mappings_with_labels(start_cui, cui2_to_cui1, cui_to_str):
    """
    Return all CUIs reachable from `start_cui` (any depth) and their labels.

    Parameters
    ----------
    start_cui : str
        The root CUI to traverse from (will be included in `all_ids`).
    cui2_to_cui1 : dict[str, list[str]]
        Mapping dictionary where each CUI may map to one or more CUIs.
    cui_to_str : dict[str, str]
        Dictionary mapping CUI IDs to human-readable string labels.

    Returns
    -------
    all_ids : set[str]
        Set of all reachable CUIs, including `start_cui`.
    all_pairs : list[tuple[str, str]]
        List of (CUI, label) pairs for all reachable CUIs,
        **excluding the starting CUI**.
    """
    visited = set()
    stack = [start_cui]

    while stack:
        cui = stack.pop()
        if cui in visited:
            continue
        visited.add(cui)

        if cui in cui2_to_cui1:
            for nxt in cui2_to_cui1[cui]:
                if nxt not in visited:
                    stack.append(nxt)

    # Build list of (cui, label), excluding the start CUI
    all_pairs = [
        (cui, cui_to_str[cui])
        for cui in visited
        if cui != start_cui and cui in cui_to_str
    ]

    return all_pairs


def assign_nearest_dataset_parents(
    df: pd.DataFrame,
    cui2_to_cui1: Dict[str, List[str]],
    cui_to_str: Dict[str, str],
    all_umls_ids: Set[str],
    parent_counts: Dict[str, int],
    id_column: str = "drug_umls_termid",
    tokens_column: str = "drug_umls_term_norm"
) -> pd.DataFrame:

    parent_ids: list[str] = []
    parent_labels: list[str] = []

    mapped_to_parent = defaultdict(set)      # "drug (mapping)" -> list of children

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Raw split
        raw_ids = str(row[id_column]).split("|")
        raw_tokens = str(row[tokens_column]).split("|")
    
        # Filter based on ID != -1 while keeping alignment
        input_ids = []
        input_tokens = []
        for tid, tok in zip(raw_ids, raw_tokens):
            tid = tid.strip()
            tok = tok.strip()
            if not tid or tid == "-1":
                continue
            input_ids.append(tid)
            input_tokens.append(tok)
    
        row_parents: list[str] = []
        row_labels: list[str] = []
    
        # Loop over aligned ids + tokens
        for child_token, child_id in zip(input_tokens, input_ids):
            parents = get_all_mappings_with_labels(child_id, cui2_to_cui1, cui_to_str)  # list[(parent_id, parent_label)]
            if not parents:
                continue
    
            for parent_id, parent_label in parents:
                # Skip if parent not in counts (defensive) or too many children
                if parent_id not in parent_counts:
                    continue
                if parent_counts[parent_id] > 20:
                    continue
    
                if parent_id in all_umls_ids:
                    row_parents.append(parent_id)
                    row_labels.append(parent_label)
    
                    # Use the *token* for the child, not the ID
                    key = f"{parent_label} ({parent_id})"
                    mapped_to_parent[key].add(child_token)
                        
            
        parent_ids.append("|".join(row_parents) if row_parents else "-1")
        parent_labels.append("|".join(row_labels) if row_labels else "-1")
    
    # Attach results to a copy of the DataFrame
    result = df.copy()
    result["nearest_dataset_parent_umls"] = parent_ids
    result["nearest_dataset_parent_umls_label"] = parent_labels

    return result, mapped_to_parent

In [45]:
fda_embedded_with_details_parent, mapped_to_parent_fda = assign_nearest_dataset_parents(
    fda_embedded_with_details,
    cui2_to_cui1,
    cui_to_str,
    all_umls_ids,
    parent_counts,
    id_column = "drug_umls_termid"
)

In [46]:
fda_embedded_with_details_parent.head()

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261,-1,-1
1,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",LISINOPRIL,Lisinopril,C0065374,-1,-1
2,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,CALCITRIOL,CALCITRIOL,Calcitriol,C0006674,-1,-1
3,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",CARBIDOPA,Carbidopa,C0006982,-1,-1
4,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",LEVODOPA,Levodopa,C0023570,-1,-1


In [47]:
# 1) rows that have a valid parent
mask = fda_embedded_with_details_parent["nearest_dataset_parent_umls"].ne("-1") & fda_embedded_with_details_parent["nearest_dataset_parent_umls"].notna()

# 2) make copies of those rows
parent_rows = fda_embedded_with_details_parent.loc[mask].copy()

# 3) overwrite drug fields with parent fields in the copies
parent_rows["drug_umls_termid"] = parent_rows["nearest_dataset_parent_umls"]
parent_rows["drug_umls_term_norm"] = parent_rows["nearest_dataset_parent_umls_label"]
parent_rows = parent_rows.drop(columns=["nearest_dataset_parent_umls", "nearest_dataset_parent_umls_label"])

# 4) append back to original df
fda_embedded_with_details = (
    pd.concat([fda_embedded_with_details, parent_rows], ignore_index=True)
      .drop_duplicates()
)


In [49]:
fda_embedded_with_details[fda_embedded_with_details['application_number']=="ANDA203518"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
1304,ANDA203518,TRIS PHARMA INC,ORIG,AP,20150512,NaN,2015,MORPHINE SULFATE,SOLUTION,ORAL,...,MORPHINE SULFATE,Bryant Ranch Prepack,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,[],MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
39331,ANDA203518,TRIS PHARMA INC,ORIG,AP,20150512,NaN,2015,MORPHINE SULFATE,SOLUTION,ORAL,...,MORPHINE SULFATE,Bryant Ranch Prepack,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,[],MORPHINE SULFATE,MORPHINE SULFATE,Morphine,C0026549


In [50]:
fda_embedded_with_details.shape

(49990, 27)

In [51]:
fda_embedded_with_details.to_csv(f"{out_dir}/FDA_full_AP_mapped_active_ingredients_with_parent.csv",index=False)